# 🩺 Predicción de Riesgo de Enfermedad Cardiovascular / Diabetes
**Autor:** Jorge Sánchez Hernández  
**Proyecto:** Machine Learning supervisado  
**Datos:** Encuesta de salud España 2025 *(Ministerio de Sanidad)*

---
**Índice**
1. Introducción y objetivo
2. Carga de datos
3. Análisis exploratorio (EDA)
4. Limpieza de datos
5. Preprocesamiento y división train/test
6. Modelo base: Regresión Logística
7. Random Forest + GridSearchCV
8. XGBoost + RandomizedSearchCV
9. Comparación de modelos
10. Evaluación final en test
11. Importancia de variables
12. Conclusiones

## 1. Introducción y objetivo

- ¿Qué problema de salud pública abordas?
- ¿Qué variable quieres predecir?
- ¿Por qué es importante minimizar los falsos negativos (priorizar **Recall**)?

In [2]:
# Librerías utilizadas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

# Que boost utilizamos?

import warnings
warnings.filterwarnings("ignore")

## 2. Carga de datos


In [10]:
#Tenemos los datos sin procesar en strings:
df_cod = pd.read_spss("../data/ESdEadulto_2025.sav", convert_categoricals=False)  # para el modelo
df_etiq = pd.read_spss("../data/ESdEadulto_2025.sav") 
print(df_cod["PROXY_0"].value_counts(dropna=False)) 

PROXY_0
1    20357
2     1284
Name: count, dtype: int64


In [6]:
df_cod.head()

,CCAA,IDENTHOGAR,NORDENa,SEXOa,EDADa,PROXY_0,PROXY_1,PROXY_2,PROXY_2b,PROXY_4,...,T3,FACTORADULTO,CLASE_PR,IMC,CMD1,CMD2,CMD3,INDICE_BIENESTAR,SEVERIDAD_DEPRESIVA,CUADROS_DEPRESIVOS
0,16,0000001,01,1,62.0,1,,,,,...,4,1223.417,5,2,7.14,10.0,3.33,68.0,1,3
1,16,0000002,01,1,59.0,1,,,,,...,5,1097.296,5,3,15.71,22.5,6.67,48.0,2,3
2,16,0000003,01,2,68.0,1,,,,,...,,1590.800,2,2,0.00,0.0,0.00,84.0,1,3
3,16,0000004,01,1,65.0,1,,,,,...,,1309.544,1,3,8.57,0.0,20.00,40.0,4,1
4,16,0000005,01,1,37.0,1,,,,,...,,786.611,3,2,0.00,0.0,0.00,60.0,2,3


In [7]:
df_etiq.head()

,CCAA,IDENTHOGAR,NORDENa,SEXOa,EDADa,PROXY_0,PROXY_1,PROXY_2,PROXY_2b,PROXY_4,...,T3,FACTORADULTO,CLASE_PR,IMC,CMD1,CMD2,CMD3,INDICE_BIENESTAR,SEVERIDAD_DEPRESIVA,CUADROS_DEPRESIVOS
0,País Vasco,0000001,01,Hombre,62.0,Sí,,,,,...,20 horas o más a la semana pero menos de 30,1223.417,Trabajadores/as cualificados/as del sector pri...,Normopeso,7.14,10.0,3.33,68.0,Ninguna,Sin cuadro depresivo alguno
1,País Vasco,0000002,01,Hombre,59.0,Sí,,,,,...,30 horas o más a la semana pero menos de 40,1097.296,Trabajadores/as cualificados/as del sector pri...,Sobrepeso,15.71,22.5,6.67,48.0,Leve,Sin cuadro depresivo alguno
2,País Vasco,0000003,01,Mujer,68.0,Sí,,,,,...,,1590.800,Directores/as y gerentes de establecimientos d...,Normopeso,0.0,0.0,0.0,84.0,Ninguna,Sin cuadro depresivo alguno
3,País Vasco,0000004,01,Hombre,65.0,Sí,,,,,...,,1309.544,Directores/as y gerentes de establecimientos d...,Sobrepeso,8.57,0.0,20.0,40.0,Moderadamente Grave,Cuadro depresivo mayor
4,País Vasco,0000005,01,Hombre,37.0,Sí,,,,,...,,786.611,Ocupaciones intermedias y trabajadores/as por ...,Normopeso,0.0,0.0,0.0,60.0,Leve,Sin cuadro depresivo alguno


In [9]:


#Creamos un diccionario para que las variables tengan nombres que podamos interpretar correctamente. 

nombres = {
    # La variable objetivo que luego se combinará en 'ecv' --> enfermedad cardiovascular)
    "C5a_2": "infarto", "C5a_3": "angina_y_otras_coronarias",
    "C5a_4": "otra_cardiopatia", "C5a_24": "ictus",

    # peso muestral (solo para el EDA, no es predictor)
    "FACTORADULTO": "factor",

    # demografía y socioeconómicas
    "EDADa": "edad", "SEXOa": "sexo", "NIVEST": "nivel_estudios",
    "CLASE_PR": "clase_social", "CCAA": "ccaa",

    # cuerpo (luego se calcula 'imc') --> poder añadir la del fichero de normopeso etc
    "N1": "altura_cm", "N2": "peso_kg", "IMC": "imc_cat",

    # antecedentes clínicos de riesgo
    "C5a_1": "tension_alta", "C5a_15": "colesterol_alto", "C5a_12": "diabetes",
    "C5a_11": "epoc", "C5a_30": "problemas_rinon", "C5a_25": "migrana",
    "C5a_27": "tumor_maligno",

    # salud mental
    "C5a_21": "depresion", "C5a_22": "ansiedad", "C5a_23": "otros_problemas_mentales",
    "SEVERIDAD_DEPRESIVA": "severidad_depresiva",
    
    # tabaco
    "Q1": "fuma", "Q2": "cigarrillos_dia", "Q6": "humo_ajeno", "Q8": "vapeo",

    # alcohol
    "R1": "frec_alcohol", "R3": "frec_atracon_alcohol",

    # ejercicio y sedentarismo (luego se calcula 'horas_sentado')
    "O1": "actividad_trabajo", "O2": "ejercicio_ocio", "O3": "dias_camina",
    "O7": "dias_deporte", "O9": "dias_fuerza",
    "O10_1": "sentado_h", "O10_2": "sentado_min",

    # dieta
    "P1_1": "dieta_fruta", "P1_9": "dieta_verdura", "P1_10": "dieta_legumbres",
    "P1_6": "dieta_pescado", "P1_2": "dieta_carne_procesada", "P1_12": "dieta_dulces",
    "P1_13": "dieta_refrescos", "P1_14": "dieta_comida_rapida", "P1_15": "dieta_aperitivos",

    # apoyo social
    "S1": "apoyo_personas",
}

# 1. Selección, paso a numérico, filtro y renombrado
df = df_cod[["PROXY_0"] + list(nombres.keys())].copy()
df = df.apply(pd.to_numeric, errors="coerce")        # los blancos pasan a NaN
df = df[df["PROXY_0"] == 1].drop(columns="PROXY_0")
df = df.rename(columns=nombres)
print(df.shape)                                       # unas 20.357 filas

# 2. Blancos por flujo (ANTES de convertir los "No contesta" en NaN)
# R3 en blanco = no bebió en 12 meses: copiamos el código de R1 (8 = lo dejó, 9 = nunca)
mask = df["frec_atracon_alcohol"].isna() & df["frec_alcohol"].isin([8, 9])
df.loc[mask, "frec_atracon_alcohol"] = df.loc[mask, "frec_alcohol"]
# Solo si usas las opcionales de medicación: en blanco = no tomó medicina (2 = No)
# df[["toma_antidepresivos", "toma_tranquilizantes"]] = df[["toma_antidepresivos", "toma_tranquilizantes"]].fillna(2)

# 3. Grupos de columnas
cols_ecv = ["infarto", "angina_y_otras_coronarias", "otra_cardiopatia", "ictus"]
cols_enfermedad = ["tension_alta", "colesterol_alto", "diabetes", "epoc", "problemas_rinon",
                   "migrana", "tumor_maligno", "depresion", "ansiedad", "otros_problemas_mentales"]
cols_dieta = ["dieta_fruta", "dieta_verdura", "dieta_legumbres", "dieta_pescado",
              "dieta_carne_procesada", "dieta_dulces", "dieta_refrescos",
              "dieta_comida_rapida", "dieta_aperitivos"]

# 4. "No contesta" -> NaN (cada grupo tiene su código)
cols_9 = (cols_ecv + cols_enfermedad + cols_dieta +
          ["clase_social", "severidad_depresiva", "imc_cat", "fuma", "humo_ajeno", "vapeo",
           "actividad_trabajo", "ejercicio_ocio", "dias_camina", "dias_deporte",
           "dias_fuerza", "apoyo_personas"])
cols_99 = ["nivel_estudios", "frec_alcohol", "frec_atracon_alcohol",
           "cigarrillos_dia", "sentado_h", "sentado_min"]
cols_999 = ["edad", "altura_cm", "peso_kg"]
df[cols_9] = df[cols_9].replace(9, np.nan)
df[cols_99] = df[cols_99].replace(99, np.nan)
df[cols_999] = df[cols_999].replace(999, np.nan)

# 5. Variable objetivo
df = df.dropna(subset=cols_ecv)
df["ecv"] = (df[cols_ecv] == 1).any(axis=1).astype(int)
df = df.drop(columns=cols_ecv)
print(df["ecv"].value_counts(normalize=True).round(3))   # ~10 % de ECV

# 6. Sí/No (1/2) -> 1/0, conservando los NaN
for c in cols_enfermedad:
    df[c] = df[c].map({1: 1, 2: 0})
df["mujer"] = df["sexo"].map({1: 0, 2: 1})

# 7. Variables nuevas
df["imc"] = df["peso_kg"] / (df["altura_cm"] / 100) ** 2
df["horas_sentado"] = df["sentado_h"] + df["sentado_min"] / 60
df = df.drop(columns=["sexo", "altura_cm", "peso_kg", "sentado_h", "sentado_min"])

(20357, 47)
ecv
0    0.9
1    0.1
Name: proportion, dtype: float64


In [4]:
df.head()

,factor,edad,nivel_estudios,clase_social,ccaa,tension_alta,colesterol_alto,diabetes,epoc,problemas_rinon,...,dieta_carne_procesada,dieta_dulces,dieta_refrescos,dieta_comida_rapida,dieta_aperitivos,apoyo_personas,ecv,mujer,imc,horas_sentado
0,1223.417,62.0,9.0,5.0,16,0.0,0.0,0.0,0.0,0.0,...,3.0,1.0,6.0,6.0,6.0,1.0,0,0,19.692665,1.5
1,1097.296,59.0,6.0,5.0,16,0.0,0.0,0.0,0.0,0.0,...,4.0,3.0,3.0,6.0,5.0,2.0,0,0,26.093275,3.0
2,1590.800,68.0,7.0,2.0,16,0.0,1.0,0.0,0.0,0.0,...,6.0,5.0,6.0,5.0,5.0,4.0,0,1,22.347782,4.0
3,1309.544,65.0,5.0,1.0,16,0.0,1.0,0.0,0.0,0.0,...,2.0,1.0,6.0,5.0,6.0,4.0,1,0,29.012346,5.0
4,786.611,37.0,9.0,3.0,16,0.0,0.0,0.0,0.0,1.0,...,4.0,3.0,6.0,4.0,5.0,3.0,0,0,20.528922,6.0


In [ ]:
import pyreadstat
import pandas as pd

# Leer datos y metadatos integrados
df, meta = pyreadstat.read_sav("../data/ESdEadulto_2025.sav", apply_value_formats=True)

# Crear una tabla con el código de cada columna y su pregunta completa
diccionario = pd.DataFrame({
    "columna": meta.column_names,
    "pregunta": meta.column_labels
})

# Buscar fácilmente las columnas de enfermedades cardiovasculares, diabetes o IMC
filtro = diccionario["pregunta"].str.contains("infarto|corazón|ictus|diabetes|masa corporal|fuma|fruta", case=False, na=False)
print(diccionario[filtro].to_string(index=False))


Empty DataFrame
Columns: [columna, pregunta]
Index: []


In [ ]:
print(df.columns)

Index(['CCAA', 'IDENTHOGAR', 'NORDENa', 'SEXOa', 'EDADa', 'PROXY_0', 'PROXY_1',
       'PROXY_2', 'PROXY_2b', 'PROXY_4',
       ...
       'T3', 'FACTORADULTO', 'CLASE_PR', 'IMC', 'CMD1', 'CMD2', 'CMD3',
       'INDICE_BIENESTAR', 'SEVERIDAD_DEPRESIVA', 'CUADROS_DEPRESIVOS'],
      dtype='str', length=446)


In [ ]:
nombres = {
    # objetivo (luego se combinan en 'ecv')
    "C5a_2": "infarto", "C5a_3": "angina_coronaria",
    "C5a_4": "otra_cardiopatia", "C5a_24": "ictus",

    # peso muestral (solo para el EDA, no es predictor)
    "FACTORADULTO": "factor",

    # demografía y socioeconómicas
    "EDADa": "edad", "SEXOa": "sexo", "NIVEST": "nivel_estudios",
    "CLASE_PR": "clase_social", "CCAA": "ccaa",

    # cuerpo (luego se calcula 'imc')
    "N1": "altura_cm", "N2": "peso_kg",

    # antecedentes clínicos
    "C5a_1": "tension_alta", "C5a_15": "colesterol_alto", "C5a_12": "diabetes",
    "C5a_11": "epoc", "C5a_30": "problemas_rinon", "C5a_25": "migrana",
    "C5a_27": "tumor_maligno",

    # salud mental
    "C5a_21": "depresion", "C5a_22": "ansiedad", "C5a_23": "otros_problemas_mentales",
    "SEVERIDAD_DEPRESIVA": "severidad_depresiva",
    # opcionales, descomenta para probarlas
    # "K3_14a": "toma_antidepresivos", "K3_7a": "toma_tranquilizantes",

    # tabaco
    "Q1": "fuma", "Q2": "cigarrillos_dia", "Q6": "humo_ajeno", "Q8": "vapeo",

    # alcohol
    "R1": "frec_alcohol", "R3": "frec_atracon_alcohol",

    # ejercicio y sedentarismo (luego se calcula 'horas_sentado')
    "O1": "actividad_trabajo", "O2": "ejercicio_ocio", "O3": "dias_camina",
    "O7": "dias_deporte", "O9": "dias_fuerza",
    "O10_1": "sentado_h", "O10_2": "sentado_min",

    # dieta
    "P1_1": "dieta_fruta", "P1_9": "dieta_verdura", "P1_10": "dieta_legumbres",
    "P1_6": "dieta_pescado", "P1_2": "dieta_carne_procesada", "P1_12": "dieta_dulces",
    "P1_13": "dieta_refrescos", "P1_14": "dieta_comida_rapida", "P1_15": "dieta_aperitivos",

    # apoyo social
    "S1": "apoyo_personas",
}

# 1. Comprobar que todas las columnas existen en df
cols_originales = list(nombres.keys())
faltan = [c for c in cols_originales + ["PROXY_0"] if c not in df.columns]
print("Columnas que faltan en df:", faltan)

# 2. Quedarse solo con las columnas elegidas
df = df[["PROXY_0"] + cols_originales].copy()

# 3. Filtrar a quienes responden por sí mismos y quitar PROXY_0
df = df[df["PROXY_0"] == "Sí"].drop(columns="PROXY_0")

# 4. Renombrar
df = df.rename(columns=nombres)

print(df.shape)
df.head()

Columnas que faltan en df: []
(0, 46)


,infarto,angina_coronaria,otra_cardiopatia,ictus,factor,edad,sexo,nivel_estudios,clase_social,ccaa,...,dieta_fruta,dieta_verdura,dieta_legumbres,dieta_pescado,dieta_carne_procesada,dieta_dulces,dieta_refrescos,dieta_comida_rapida,dieta_aperitivos,apoyo_personas


## 3. Análisis exploratorio (EDA)

In [ ]:
for col in df.columns:
    print(col)


CCAA
IDENTHOGAR
NORDENa
SEXOa
EDADa
PROXY_0
PROXY_1
PROXY_2
PROXY_2b
PROXY_4
PROXY_5
PROXY_6
A1a
A1c
A1e
A2a_1
A2a_2
A2a_3
A3
A4
A5
NIVEST
B1
B2
B3_2
B4_2
B5
B6
B7
B8
B9a
B9b
B10
B11
B12
B13a_2
B13b_2
B14a_2
B14b_2
B15
C1
C2
C3a1
C3a2
C3b
C4
C5a_1
C5b_1
C5c_1
C5a_2
C5b_2
C5c_2
C5d_2
C5a_3
C5b_3
C5c_3
C5a_4
C5b_4
C5c_4
C5a_5
C5b_5
C5c_5
C5a_6
C5b_6
C5c_6
C5a_7
C5b_7
C5c_7
C5a_8
C5b_8
C5c_8
C5a_9
C5b_9
C5c_9
C5a_10
C5b_10
C5c_10
C5a_11
C5b_11
C5c_11
C5a_12
C5b_12
C5c_12
C5a_13
C5b_13
C5c_13
C5a_14
C5b_14
C5c_14
C5a_15
C5b_15
C5c_15
C5a_16
C5b_16
C5c_16
C5a_17
C5b_17
C5c_17
C5a_18
C5b_18
C5c_18
C5a_19
C5b_19
C5c_19
C5a_20
C5b_20
C5c_20
C5a_21
C5b_21
C5c_21
C5a_22
C5b_22
C5c_22
C5a_23
C5b_23
C5c_23
C5a_24
C5b_24
C5c_24
C5d_24
C5a_25
C5b_25
C5c_25
C5a_26
C5b_26
C5c_26
C5a_27
C5b_27
C5c_27
C5a_28
C5b_28
C5c_28
C5a_29
C5b_29
C5c_29
C5a_30
C5b_30
C5c_30
C5a_31
C5b_31
C5c_31
C5a_32
C5b_32
C5c_32
C5a_33
C5b_33
C5c_33
D1_1
D1_2
D1_3
D2
E1
E2
F1
F2
F3
F4
F5
F6
F7
F8
F8a
F9
G1_1
G1_2
G1_3
G1_4
G1_5

In [ ]:
df.describe()

In [ ]:
# Valores nulos (nº y %)
nulos = pd.DataFrame({"nulos": df.isnull().sum(),
                      "porcentaje": (df.isnull().mean() * 100).round(2)})
nulos[nulos["nulos"] > 0].sort_values("porcentaje", ascending=False)

In [ ]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())

### 3.1 Variable objetivo
*Define qué columna(s) usas. Si combinas enfermedad cardiovascular y diabetes en una sola variable, explica el criterio.*

In [ ]:
# TODO: ajusta los nombres de columna de tu dataset
# Opción A: ya existe una columna binaria
# df["target"] = df["NOMBRE_COLUMNA"]

# Opción B: combinar varias columnas (1 = tiene alguna de las dos)
# df["target"] = ((df["diabetes"] == 1) | (df["cardiovascular"] == 1)).astype(int)

# print(df["target"].value_counts())
# print(df["target"].value_counts(normalize=True).round(3))

In [ ]:
# Distribución de la variable objetivo (descomenta cuando exista 'target')
# sns.countplot(x="target", data=df)
# plt.title("Distribución de la variable objetivo")
# plt.show()

### 3.2 Distribución de variables y relación con el objetivo

In [ ]:
# Variables numéricas: histogramas
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols].hist(figsize=(14, 10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas: nº de categorías
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(col, "->", df[col].nunique(), "categorías")

In [ ]:
# Ejemplo: relación de una variable con el objetivo (cambia 'edad' por la tuya)
# sns.boxplot(x="target", y="edad", data=df)
# plt.show()

# Ejemplo categórica:
# sns.countplot(x="tabaco", hue="target", data=df)
# plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Matriz de correlación")
plt.show()

**Observaciones del EDA:** *(escribe aquí lo que has visto: desbalanceo, variables con más correlación, outliers, etc.)*

## 4. Limpieza de datos

In [ ]:
# Eliminar duplicados
df = df.drop_duplicates()

# Eliminar columnas que no sirven (identificadores, etc.)
# df = df.drop(columns=["id"])

# Nulos: ejemplos sencillos
# Numéricas -> mediana
# df["imc"] = df["imc"].fillna(df["imc"].median())
# Categóricas -> moda
# df["tabaco"] = df["tabaco"].fillna(df["tabaco"].mode()[0])
# O eliminar filas con nulos:
# df = df.dropna()

print(df.shape)
print("Nulos restantes:", df.isnull().sum().sum())

*Justifica aquí las decisiones de limpieza (por qué mediana, por qué eliminas filas, etc.).*

## 5. Preprocesamiento y división train/test

In [ ]:
# Separar X e y
X = df.drop(columns=["target"])
y = df["target"]

# Codificar variables categóricas
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

In [ ]:
# División train/test (estratificada para mantener la proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# Escalado (solo para Regresión Logística). Se ajusta con train y se aplica a test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

**Desbalanceo de clases:** si la clase positiva es minoritaria, usamos `class_weight="balanced"` en los modelos (y `scale_pos_weight` en XGBoost) para que el modelo no ignore a los enfermos.

In [ ]:
# Peso para XGBoost = nº negativos / nº positivos
peso_pos = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(peso_pos, 2))

## 6. Modelo base: Regresión Logística

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)

# Validación cruzada (priorizamos recall)
cv_recall = cross_val_score(log_reg, X_train_sc, y_train, cv=5, scoring="recall")
cv_f1 = cross_val_score(log_reg, X_train_sc, y_train, cv=5, scoring="f1")

print("Recall CV:", cv_recall.round(3), "-> media", cv_recall.mean().round(3))
print("F1 CV:    ", cv_f1.round(3), "-> media", cv_f1.mean().round(3))

In [ ]:
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

print(classification_report(y_test, y_pred_lr))

**Comentario:** *(¿qué tal funciona el modelo base? ¿Recall alto pero precisión baja?)*

## 7. Random Forest + GridSearchCV

In [ ]:
rf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5, 10]
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring="recall", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Mejores parámetros:", grid_rf.best_params_)
print("Mejor recall CV:", round(grid_rf.best_score_, 3))

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

## 8. XGBoost + RandomizedSearchCV

In [ ]:
xgb = XGBClassifier(scale_pos_weight=peso_pos, eval_metric="logloss", random_state=RANDOM_STATE)

param_dist_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0]
}

rand_xgb = RandomizedSearchCV(xgb, param_dist_xgb, n_iter=10, cv=5, scoring="recall",
                              random_state=RANDOM_STATE, n_jobs=-1)
rand_xgb.fit(X_train, y_train)

print("Mejores parámetros:", rand_xgb.best_params_)
print("Mejor recall CV:", round(rand_xgb.best_score_, 3))

In [ ]:
best_xgb = rand_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

## 9. Comparación de modelos

In [ ]:
modelos = {
    "Regresión Logística": (y_pred_lr, log_reg.predict_proba(X_test_sc)[:, 1]),
    "Random Forest": (y_pred_rf, best_rf.predict_proba(X_test)[:, 1]),
    "XGBoost": (y_pred_xgb, best_xgb.predict_proba(X_test)[:, 1]),
}

resultados = []
for nombre, (pred, proba) in modelos.items():
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, proba),
    })

tabla = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabla

In [ ]:
tabla[["Recall", "F1", "ROC AUC"]].plot(kind="bar", figsize=(9, 5))
plt.title("Comparación de modelos")
plt.ylabel("Métrica")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

## 10. Evaluación final en test
*Elige el mejor modelo según Recall y F1 y justifica por qué.*

In [ ]:
# Cambia por el modelo ganador
mejor_modelo = best_xgb
y_pred_final = y_pred_xgb

cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(cm, display_labels=["Sin enfermedad", "Con enfermedad"]).plot(cmap="Blues")
plt.title("Matriz de confusión - modelo final")
plt.show()

**Interpretación:** *(cuántos falsos negativos hay, qué implicaría en la práctica, etc.)*

## 11. Importancia de variables

In [ ]:
importancias = pd.Series(mejor_modelo.feature_importances_, index=X_train.columns)
top = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=top.values, y=top.index)
plt.title("Top 15 variables más importantes")
plt.xlabel("Importancia")
plt.show()

**Interpretación desde salud pública:** *(qué factores de riesgo destacan: edad, IMC, tabaco, actividad física... ¿tiene sentido clínico?)*

## 12. Conclusiones
- **Resultados:** *(mejor modelo y métricas)*
- **Factores de riesgo más relevantes:** *...*
- **Limitaciones:** *(datos autodeclarados, desbalanceo, variables que faltan...)*
- **Próximos pasos:** *(más modelos, ajustar umbral de decisión, más datos...)*